This notebook shows the success of our modles on the single channel case and the two and four channel cases when a simple mixing matrix is used

# Generating The Mixing Matrix

In [ ]:
import numpy as np

def _sample_phase_change_matrix(self, n_rx: int, phase_change_deg: float = 5.0) -> np.ndarray:
    # Convert degrees passed in to radians
    phase_step = np.deg2rad(phase_change_deg)
    antenna_phases = np.arange(n_rx) * phase_step

    phases = np.zeros((n_rx, 2), dtype=np.float64)
    # positive case represents signal of interest coming from the left side
    phases[:, 0] = antenna_phases          # source A: 0, Δφ, 2Δφ, ...
    # negative case represents interferer coming from the right side
    phases[:, 1] = -antenna_phases         # source B: 0, -Δφ, -2Δφ, ...

    return np.exp(1j * phases).astype(np.complex128)

## Creating The Baseband Mixture S(t)

In [ ]:
# Multi-channel receive case
H = self._sample_phase_change_matrix(
    n_rx=mix_cfg.n_rx,
    phase_change_deg=mix_cfg.phase_shift_deg
)

# Stack sources as (2, T)
sources = np.vstack([
    s_soi,
    mix_cfg.alpha * s_int,
])

# Linear mixture at all receivers: (n_rx, 2) @ (2, T) -> (n_rx, T)
signal = H @ sources

if noise_cfg.enabled:
    noise = self.generate_noise(signal, mix_cfg.snr_db)
    mixture = signal + noise
else:
    noise = np.zeros_like(signal)
    mixture = signal

# Testing Our Separators

Start with setup
- Import everything you need
- Set config values from config.py

In [1]:
import numpy as np
import torch
import matplotlib.pyplot as plt

from config import ExperimentConfig
from utils.data_utils.generator import RFMixtureGenerator, QPSKConfig, NoiseConfig, MixtureConfig
from utils.model_utils.symbol_utils import rrc_taps, recover_symbols_from_waveform, symbol_accuracy
from utils.model_utils.conversion_helpers import complex_to_2ch, complex_matrix_to_iq_channels

from networks.iq_cnn_separator import IQCNNSeparator

In [2]:
gen = RFMixtureGenerator(seed=0)

qpsk_cfg_soi = QPSKConfig(
    n_symbols=ExperimentConfig.num_symbols,
    samples_per_symbol=ExperimentConfig.samples_per_symbol,
    rolloff=ExperimentConfig.rolloff,
    rrc_span_symbols=ExperimentConfig.rrc_span_symbols,
)

qpsk_cfg_int = QPSKConfig(
    n_symbols=ExperimentConfig.num_symbols,
    samples_per_symbol=ExperimentConfig.samples_per_symbol,
    rolloff=ExperimentConfig.rolloff,
    rrc_span_symbols=ExperimentConfig.rrc_span_symbols,
)

noise_cfg = NoiseConfig(
    enabled=ExperimentConfig.noise_enabled
)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### Single Channel

Results For **Entire Dataset** [Train: 10,000 | Test: 1,000 | 20]

IQ_CNN | Train Sym Acc: 1.000 | Val Sym Acc: 1.000

#### Single Example Generated On The Fly Test

Set To Single Channel and Load The Model

In [3]:
mix_cfg = MixtureConfig(
    alpha=ExperimentConfig.noise_alpha,
    snr_db=ExperimentConfig.snr_db,
    n_rx=1,
    random_phase=ExperimentConfig.random_phase,
    phase_shift_deg=ExperimentConfig.phase_shift_deg,
)

# Load the model
model = IQCNNSeparator(in_ch=2, out_ch=4).to(device)
model.load_state_dict(torch.load("pytorch_models/IQ_CNN_1_channel.pt", map_location=device))
model.eval()

/tmp/ipykernel_2864418/167568541.py:11: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("pytorch_models/IQ_CNN_1_channel.pt", map_location=dev

IQCNNSeparator(
  (unet): MultichannelUNetSeparator(
    (input_proj): Conv1d(2, 32, kernel_size=(1,), stride=(1,), bias=False)
    (input_gn): GroupNorm(2, 32, eps=1e-05, affine=True)
    (input_act): GELU(approximate='none')
    (down1): DownsampleBlock(
      (block): ConvBlock1D(
        (conv1): Conv1d(32, 32, kernel_size=(5,), stride=(2,), padding=(2,), bias=False)
        (gn1): GroupNorm(2, 32, eps=1e-05, affine=True)
        (conv2): Conv1d(32, 32, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
        (gn2): GroupNorm(2, 32, eps=1e-05, affine=True)
        (dropout): Identity()
        (act): GELU(approximate='none')
      )
    )
    (down2): DownsampleBlock(
      (block): ConvBlock1D(
        (conv1): Conv1d(32, 64, kernel_size=(5,), stride=(2,), padding=(2,), bias=False)
        (gn1): GroupNorm(4, 64, eps=1e-05, affine=True)
        (conv2): Conv1d(64, 64, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
        (gn2): GroupNorm(4, 64, eps=1e-05, affine

Create a single example

In [4]:
# Create a single example on the fly
ex = gen.generate_mixture(qpsk_cfg_soi, qpsk_cfg_int, noise_cfg, mix_cfg)

Pass example through model

In [5]:
# Convert mixture to model input
if mix_cfg.n_rx == 1:
    x = complex_to_2ch(ex["mixture"])
else:
    x = complex_matrix_to_iq_channels(ex["mixture"])

# Convert to tensor so it can be used by pytorch
x_tensor = torch.from_numpy(x).float().unsqueeze(0).to(device)

# Run model
with torch.no_grad():
    pred = model(x_tensor).squeeze(0).cpu().numpy()

# Convert outputs back to complex waveforms
pred_a = pred[0] + 1j * pred[1]
pred_b = pred[2] + 1j * pred[3]

# Build matched filter
rrc = rrc_taps(
    sps=ExperimentConfig.samples_per_symbol,
    beta=ExperimentConfig.rolloff,
    span_symbols=ExperimentConfig.rrc_span_symbols,
)

# Recover symbols
rec_a = recover_symbols_from_waveform(
    pred_a, rrc, ExperimentConfig.samples_per_symbol, len(ex["symbols_a"])
)
rec_b = recover_symbols_from_waveform(
    pred_b, rrc, ExperimentConfig.samples_per_symbol, len(ex["symbols_b"])
)

# True symbols
true_a = ex["symbols_a"]
true_b = ex["symbols_b"]

# Direct assignment
n_aa = min(len(rec_a), len(true_a))
n_bb = min(len(rec_b), len(true_b))
direct_score = 0.5 * (
    symbol_accuracy(rec_a[:n_aa], true_a[:n_aa]) +
    symbol_accuracy(rec_b[:n_bb], true_b[:n_bb])
)

# Swapped assignment
n_ab = min(len(rec_a), len(true_b))
n_ba = min(len(rec_b), len(true_a))
swap_score = 0.5 * (
    symbol_accuracy(rec_a[:n_ab], true_b[:n_ab]) +
    symbol_accuracy(rec_b[:n_ba], true_a[:n_ba])
)

print("Direct symbol accuracy:", max(direct_score, swap_score))

Direct symbol accuracy: 1.0


### Two Channel

Results For **Entire Dataset** Compared To FastICA [Train: 10,000 | Test: 1,000 | 20]

FastICA | Train Sym Acc: 0.3627 | Val Sym Acc: 0.3631

IQ_CNN  | Train Sym Acc: 1.0000 | Val Sym Acc: 1.0000


#### Single Example Generated On The Fly Test

Set To two Channel and Load The Model

In [6]:
mix_cfg = MixtureConfig(
    alpha=ExperimentConfig.noise_alpha,
    snr_db=ExperimentConfig.snr_db,
    n_rx=2,
    random_phase=ExperimentConfig.random_phase,
    phase_shift_deg=ExperimentConfig.phase_shift_deg,
)

# Load the model
model = IQCNNSeparator(in_ch=4, out_ch=4).to(device)
model.load_state_dict(torch.load("pytorch_models/IQ_CNN_2_channel.pt", map_location=device))
model.eval()

/tmp/ipykernel_2864418/1118340248.py:11: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("pytorch_models/IQ_CNN_2_channel.pt", map_location=de

IQCNNSeparator(
  (unet): MultichannelUNetSeparator(
    (input_proj): Conv1d(4, 32, kernel_size=(1,), stride=(1,), bias=False)
    (input_gn): GroupNorm(2, 32, eps=1e-05, affine=True)
    (input_act): GELU(approximate='none')
    (down1): DownsampleBlock(
      (block): ConvBlock1D(
        (conv1): Conv1d(32, 32, kernel_size=(5,), stride=(2,), padding=(2,), bias=False)
        (gn1): GroupNorm(2, 32, eps=1e-05, affine=True)
        (conv2): Conv1d(32, 32, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
        (gn2): GroupNorm(2, 32, eps=1e-05, affine=True)
        (dropout): Identity()
        (act): GELU(approximate='none')
      )
    )
    (down2): DownsampleBlock(
      (block): ConvBlock1D(
        (conv1): Conv1d(32, 64, kernel_size=(5,), stride=(2,), padding=(2,), bias=False)
        (gn1): GroupNorm(4, 64, eps=1e-05, affine=True)
        (conv2): Conv1d(64, 64, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
        (gn2): GroupNorm(4, 64, eps=1e-05, affine

Create a single example

In [7]:
# Create a single example on the fly
ex = gen.generate_mixture(qpsk_cfg_soi, qpsk_cfg_int, noise_cfg, mix_cfg)

Pass example through model

In [8]:
# Convert mixture to model input
if mix_cfg.n_rx == 1:
    x = complex_to_2ch(ex["mixture"])
else:
    x = complex_matrix_to_iq_channels(ex["mixture"])

# Convert to tensor so it can be used by pytorch
x_tensor = torch.from_numpy(x).float().unsqueeze(0).to(device)

# Run model
with torch.no_grad():
    pred = model(x_tensor).squeeze(0).cpu().numpy()

# Convert outputs back to complex waveforms
pred_a = pred[0] + 1j * pred[1]
pred_b = pred[2] + 1j * pred[3]

# Build matched filter
rrc = rrc_taps(
    sps=ExperimentConfig.samples_per_symbol,
    beta=ExperimentConfig.rolloff,
    span_symbols=ExperimentConfig.rrc_span_symbols,
)

# Recover symbols
rec_a = recover_symbols_from_waveform(
    pred_a, rrc, ExperimentConfig.samples_per_symbol, len(ex["symbols_a"])
)
rec_b = recover_symbols_from_waveform(
    pred_b, rrc, ExperimentConfig.samples_per_symbol, len(ex["symbols_b"])
)

# True symbols
true_a = ex["symbols_a"]
true_b = ex["symbols_b"]

# Direct assignment
n_aa = min(len(rec_a), len(true_a))
n_bb = min(len(rec_b), len(true_b))
direct_score = 0.5 * (
    symbol_accuracy(rec_a[:n_aa], true_a[:n_aa]) +
    symbol_accuracy(rec_b[:n_bb], true_b[:n_bb])
)

# Swapped assignment
n_ab = min(len(rec_a), len(true_b))
n_ba = min(len(rec_b), len(true_a))
swap_score = 0.5 * (
    symbol_accuracy(rec_a[:n_ab], true_b[:n_ab]) +
    symbol_accuracy(rec_b[:n_ba], true_a[:n_ba])
)

print("Direct Symbol Accuracy:", max(direct_score, swap_score))

Direct Symbol Accuracy: 1.0


### Four Channel

Results For **Entire Dataset** Compared To FastICA [Train: 10,000 | Test: 1,000 | 20 epochs]

FastICA | Train Sym Acc: 0.3701 | Val Sym Acc: 0.3679 

IQ_CNN  | Train Sym Acc: 1.0000 | Val Sym Acc: 1.0000

#### Single Example Generated On The Fly Test

Set To Four Channel and Load The Model

In [9]:
mix_cfg = MixtureConfig(
    alpha=ExperimentConfig.noise_alpha,
    snr_db=ExperimentConfig.snr_db,
    n_rx=4,
    random_phase=ExperimentConfig.random_phase,
    phase_shift_deg=ExperimentConfig.phase_shift_deg,
)

# Load the model
model = IQCNNSeparator(in_ch=8, out_ch=4).to(device)
model.load_state_dict(torch.load("pytorch_models/IQ_CNN_4_channel.pt", map_location=device))
model.eval()

/tmp/ipykernel_2864418/4025415060.py:11: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("pytorch_models/IQ_CNN_4_channel.pt", map_location=de

IQCNNSeparator(
  (unet): MultichannelUNetSeparator(
    (input_proj): Conv1d(8, 32, kernel_size=(1,), stride=(1,), bias=False)
    (input_gn): GroupNorm(2, 32, eps=1e-05, affine=True)
    (input_act): GELU(approximate='none')
    (down1): DownsampleBlock(
      (block): ConvBlock1D(
        (conv1): Conv1d(32, 32, kernel_size=(5,), stride=(2,), padding=(2,), bias=False)
        (gn1): GroupNorm(2, 32, eps=1e-05, affine=True)
        (conv2): Conv1d(32, 32, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
        (gn2): GroupNorm(2, 32, eps=1e-05, affine=True)
        (dropout): Identity()
        (act): GELU(approximate='none')
      )
    )
    (down2): DownsampleBlock(
      (block): ConvBlock1D(
        (conv1): Conv1d(32, 64, kernel_size=(5,), stride=(2,), padding=(2,), bias=False)
        (gn1): GroupNorm(4, 64, eps=1e-05, affine=True)
        (conv2): Conv1d(64, 64, kernel_size=(5,), stride=(1,), padding=(2,), bias=False)
        (gn2): GroupNorm(4, 64, eps=1e-05, affine

Create a single example

In [10]:
# Create a single example on the fly
ex = gen.generate_mixture(qpsk_cfg_soi, qpsk_cfg_int, noise_cfg, mix_cfg)

Pass example through model

In [11]:
# Convert mixture to model input
if mix_cfg.n_rx == 1:
    x = complex_to_2ch(ex["mixture"])
else:
    x = complex_matrix_to_iq_channels(ex["mixture"])

# Convert to tensor so it can be used by pytorch
x_tensor = torch.from_numpy(x).float().unsqueeze(0).to(device)

# Run model
with torch.no_grad():
    pred = model(x_tensor).squeeze(0).cpu().numpy()

# Convert outputs back to complex waveforms
pred_a = pred[0] + 1j * pred[1]
pred_b = pred[2] + 1j * pred[3]

# Build matched filter
rrc = rrc_taps(
    sps=ExperimentConfig.samples_per_symbol,
    beta=ExperimentConfig.rolloff,
    span_symbols=ExperimentConfig.rrc_span_symbols,
)

# Recover symbols
rec_a = recover_symbols_from_waveform(
    pred_a, rrc, ExperimentConfig.samples_per_symbol, len(ex["symbols_a"])
)
rec_b = recover_symbols_from_waveform(
    pred_b, rrc, ExperimentConfig.samples_per_symbol, len(ex["symbols_b"])
)

# True symbols
true_a = ex["symbols_a"]
true_b = ex["symbols_b"]

# Direct assignment
n_aa = min(len(rec_a), len(true_a))
n_bb = min(len(rec_b), len(true_b))
direct_score = 0.5 * (
    symbol_accuracy(rec_a[:n_aa], true_a[:n_aa]) +
    symbol_accuracy(rec_b[:n_bb], true_b[:n_bb])
)

# Swapped assignment
n_ab = min(len(rec_a), len(true_b))
n_ba = min(len(rec_b), len(true_a))
swap_score = 0.5 * (
    symbol_accuracy(rec_a[:n_ab], true_b[:n_ab]) +
    symbol_accuracy(rec_b[:n_ba], true_a[:n_ba])
)

print("Direct Symbol Accuracy:", max(direct_score, swap_score))

Direct Symbol Accuracy: 1.0


This markdown file shows that we are able to get 100% train and validation accuracy in the single channel, 2 channel, and 4 channel cases. This represents that the repository is in a working, usable state, and that settings can easily be changed to fit a the users use case.